# Baseline2: Adding Finegrained Labels\

- after finishing the baseline-add imgtxt source idx can dot this work.

In [ ]:
import os
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
    
# 注意路径
info_fp = '../data/hateful_memes/info.csv'

# 一定要改这个路径
fine_grained_data_dir = '../data/hateful_memes_finegrained'

In [ ]:
split = 'train'
fp = f'{fine_grained_data_dir}/{split}.json'
fine_grained_df = pd.read_json(fp, lines=True)
fine_grained_df.head()

In [ ]:
fine_grained_df.tail()

In [ ]:
print(f'fine_grained_df.shape: {fine_grained_df.shape}')

`gold` means: 机器学习和数据标注中，gold 一词通常用于指代"黄金标准"（Gold Standard），即被认为是最准确、最权威的标注或标签

In [ ]:
print((fine_grained_df['gold_pc'].apply(lambda x: len(x)) > 1).sum())
print((fine_grained_df['gold_attack'].apply(lambda x: len(x)) > 1).sum())

- 标签二值化

In [ ]:
mlb = MultiLabelBinarizer()
transformed = mlb.fit_transform(fine_grained_df['gold_attack'])
print(transformed.shape)
print(mlb.classes_)

In [ ]:
def find_unique_labels(lists, empty_replacement):
    
     if isinstance(lists, list):
          return list({item for lst in lists for item in lst})
     return [empty_replacement]
 
print((fine_grained_df['pc'].apply(find_unique_labels, empty_replacement='pc_empty').apply(lambda x: len(x)) > 1).sum())

print((fine_grained_df['attack'].apply(find_unique_labels, empty_replacement='attack_empty').apply(lambda x: len(x)) > 1).sum())

- 图像转化

分别两个类型的 label cols, 用两个不同的二值化工具来进行划分 (model fit得到：模型二值化多分类标签的能力)
> output: [xxx] one-hot encoding

```python
mlb_pc, mlb_attack = MultiLabelBinarizer(), MultiLabelBinarizer()
```

In [ ]:
def find_unique_labels(lists, empty_replacement):
    
     if isinstance(lists, list):
          return list({item for lst in lists for item in lst})
     return [empty_replacement]
 

splits = ['train', 'dev_seen', 'dev_unseen']
# splits = ['train, ']

fine_grained_dfs = []

# fine_grained_data_dir = '../data/hateful_memes_finegrained'

for split in splits:
    fp = f'{fine_grained_data_dir}/{split}.json'
    fine_grained_df = pd.read_json(fp, lines=True)
    fine_grained_dfs.append(fine_grained_df)
    
fine_grained_df = pd.concat(fine_grained_dfs)

# 
mlb_pc, mlb_attack = MultiLabelBinarizer(), MultiLabelBinarizer()
mlb_pc.fit(fine_grained_df['gold_pc'])
mlb_attack.fit(fine_grained_df['gold_attack'])

# mlb_pc / mlb_attack 这两个是 model对象

In [ ]:
splits = ['train', 'dev_seen', 'dev_unseen']
fine_grained_dfs = []

for split in splits:
    fp = f'{fine_grained_data_dir}/{split}.json'
    fine_grained_df = pd.read_json(fp, lines=True)
    # transform 'pc' like 'gold_pc'
    fine_grained_df['pc'] = fine_grained_df['pc'].apply(find_unique_labels, empty_replacement='pc_empty')
    # transform 'attack' like 'gold_attack'
    fine_grained_df['attack'] = fine_grained_df['attack'].apply(find_unique_labels, empty_replacement='attack_empty')
    # binarize 'gold_pc' and 'gold_attack'
    new_cols = [x+'_gold_pc' for x in mlb_pc.classes_]
    fine_grained_df[new_cols] = mlb_pc.transform(fine_grained_df['gold_pc'])
    new_cols = [x+'_gold_attack' for x in mlb_attack.classes_]
    fine_grained_df[new_cols] = mlb_attack.transform(fine_grained_df['gold_attack'])
    # binarize 'pc' and 'attack'
    new_cols = [x+'_pc' for x in mlb_pc.classes_]
    fine_grained_df[new_cols] = mlb_pc.transform(fine_grained_df['pc'])
    new_cols = [x+'_attack' for x in mlb_attack.classes_]
    fine_grained_df[new_cols] = mlb_attack.transform(fine_grained_df['attack'])

    fine_grained_dfs.append(fine_grained_df)

fine_grained_df = pd.concat(fine_grained_dfs)
print(fine_grained_df.shape)

fine_grained_df.head()

In [ ]:
fine_grained_df.columns

In [ ]:
cols_to_remove = ['img', 'text', 'gold_hate']
fine_grained_df = fine_grained_df.drop(columns=cols_to_remove)
fine_grained_df = fine_grained_df.rename(columns={'set_name':'split'})
print(fine_grained_df.shape)

fine_grained_df.head()

In [ ]:
fine_grained_df.columns

- 连接前面处理后的相关联性数据

In [ ]:
# info_fp = '../data/hateful_memes/info.csv'

info_df = pd.read_csv(info_fp)
print(info_df.shape)
info_df.head()

In [ ]:
# merge
info_df = pd.merge(info_df, fine_grained_df, on=['id', 'split'], how='left')

print(info_df.shape)
info_df.tail()

In [ ]:
# 保证所有 idx 的数值类型都是 int
float_cols = info_df.select_dtypes(float).columns
info_df[float_cols] = info_df.select_dtypes(float).astype('Int64')
info_df.head()

In [ ]:
# info_fine_grained label
info_df.to_csv(info_fp.replace('info', 'info_fine_grained'), index=False)

pc_columns = [col for col in info_df.columns if col.endswith('_pc') and not 'gold' in col]
attack_columns = [col for col in info_df.columns if col.endswith('_attack') and not 'gold' in col]
fine_grained_labels = pc_columns + attack_columns
print(fine_grained_labels)

In [ ]:
with open('fine_grained_labels.txt', 'w') as file:
    file.writelines([line+'\n' for line in fine_grained_labels])